# 06A — Calibration sampling sensitivity

This optional, label-free diagnostic is run when the model-fitting sampling policy changes. It compares 4, 8, 16 and uncapped-per-entity-day calibration rows across three deterministic seeds.

It does **not** read development fault labels or holdout data. The `all_per_day` option removes the entity-day cap but retains the explicitly reported global training-row guard to keep the experiment safe on notebook hardware.


## 1. Resolve the same feature and policy lineage used by Notebook 06


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    override = os.getenv("TELCO_PROJECT_ROOT")
    candidates = [Path(override).expanduser().resolve()] if override else [
        start.resolve(), *start.resolve().parents,
    ]
    if "google.colab" in sys.modules:
        candidates += [
            Path("/content/drive/MyDrive/anomaly_detection"),
            Path("/content/drive/MyDrive/telco-anomaly-detection"),
        ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the repository or set TELCO_PROJECT_ROOT")


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.scoring.sensitivity import run_sampling_sensitivity
from telco_anomaly.io import (
    file_sha256, immutable_output_directory, load_config, read_json,
    require_same, resolve_data_root, source_tree_sha256, write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
CORE_RUN_ID = os.getenv("TELCO_CORE_RUN_ID", "synthetic_pon_core_v2")
FEATURE_RUN_ID = os.getenv("TELCO_FEATURE_RUN_ID", f"{DATASET}_features_v6")
EDA_RUN_ID = os.getenv("TELCO_EDA_RUN_ID", f"{DATASET}_calibration_eda_v5")
SENSITIVITY_RUN_ID = os.getenv(
    "TELCO_SENSITIVITY_RUN_ID", f"{DATASET}_sampling_sensitivity_v1"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
FEATURE_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID
EDA_ROOT = DATA_ROOT / "eda" / DATASET / EDA_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "diagnostics" / DATASET / SENSITIVITY_RUN_ID

if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Sampling sensitivity must use the truth-unmounted run")

MODEL_CONFIG = load_config("model", project_root=PROJECT_ROOT)
ALERT_POLICY = load_config("alert_policy", project_root=PROJECT_ROOT)
feature_manifest = read_json(FEATURE_ROOT / "feature_manifest.json")
eda_decisions = read_json(EDA_ROOT / "eda_decisions.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
feature_paths = {
    name: FEATURE_ROOT / values["features"]
    for name, values in feature_manifest["partitions"].items()
}

LINEAGE = {
    "feature_manifest_sha256": file_sha256(FEATURE_ROOT / "feature_manifest.json"),
    "model_config_sha256": file_sha256(PROJECT_ROOT / "configs" / "model.yml"),
    "detector_package_sha256": source_tree_sha256(
        PROJECT_ROOT / "src" / "telco_anomaly",
        patterns=("detectors.py", "scoring/*.py", "pipeline/*.py"),
    ),
}

display(pd.Series({
    "calibration_fit": str(feature_paths["calibration_fit"]),
    "late_calibration": str(feature_paths["calibration_threshold"]),
    "output": str(OUTPUT_ROOT),
    **LINEAGE,
}, name="value").to_frame())


## 2. Run the registered sampling grid

Each fitted model is thresholded on the first half of late calibration. Incident workload is measured on the independent second half. Rank stability uses exactly the same verification rows for every configuration.


In [ ]:
policy = MODEL_CONFIG["sampling_sensitivity"]
isolation = MODEL_CONFIG["isolation_forest"]
reference = MODEL_CONFIG["robust_reference"]
rows_per_day = [
    None if str(value).lower() == "all" else int(value)
    for value in policy["rows_per_entity_day"]
]

cadences = pd.to_numeric(
    catalogue["expected_cadence_seconds"], errors="coerce"
).dropna().unique()
if len(cadences) != 1:
    raise ValueError("Sampling sensitivity requires one prepared modelling cadence")
cadence_seconds = float(cadences[0])

fit_kwargs = {
    "use_entity_reference": True,
    "catalogue": catalogue,
    "reference_exclusions": eda_decisions.get("reference_exclusions", []),
    "isolation_max_samples": int(isolation["max_samples"]),
    "isolation_max_features": float(isolation["max_features"]),
    "maximum_isolation_features_per_metric": int(
        isolation["maximum_features_per_metric"]
    ),
    "entity_reference_minimum_rows": int(reference["minimum_entity_rows"]),
    "entity_reference_minimum_days": float(reference["minimum_entity_days"]),
    "entity_reference_shrinkage_days": float(reference["shrinkage_days"]),
    "isolation_entity_minimum_rows": int(
        isolation["entity_score_calibration"]["minimum_rows"]
    ),
    "isolation_entity_scale_floor_fraction": float(
        isolation["entity_score_calibration"]["scale_floor_fraction_of_global"]
    ),
}

if OUTPUT_ROOT.exists():
    manifest = read_json(OUTPUT_ROOT / "sensitivity_manifest.json")
    require_same(manifest, **LINEAGE)
    sensitivity = pd.read_parquet(OUTPUT_ROOT / "sampling_sensitivity.parquet")
    summary = manifest["summary"]
    print("Using existing immutable sensitivity result:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        sensitivity, summary = run_sampling_sensitivity(
            feature_paths["calibration_fit"],
            feature_paths["calibration_threshold"],
            output,
            fit_kwargs=fit_kwargs,
            cadence_seconds=cadence_seconds,
            dispersion_window_seconds=float(feature_manifest["dispersion_window_seconds"]),
            cusum_allowance=float(
                ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"]
            ),
            rows_per_entity_day=rows_per_day,
            random_seeds=policy["random_seeds"],
            baseline_rows_per_entity_day=int(policy["baseline_rows_per_entity_day"]),
            baseline_seed=int(policy["baseline_seed"]),
            maximum_training_rows=int(policy["maximum_training_rows"]),
            maximum_scoring_rows=int(policy["maximum_scoring_rows"]),
            isolation_trees=int(policy["n_estimators"]),
            threshold_quantile=float(policy["threshold_quantile"]),
            threshold_block_seconds=float(ALERT_POLICY["thresholds"]["block_seconds"]),
            persistence_observations=max(
                1,
                round(
                    ALERT_POLICY["channels"]["isolation_forest_temporal"]
                    ["persistence_seconds"] / cadence_seconds
                ),
            ),
            recovery_observations=max(
                1, round(ALERT_POLICY["recovery"]["duration_seconds"] / cadence_seconds)
            ),
            recovery_threshold_fraction=float(
                ALERT_POLICY["recovery"]["threshold_fraction"]
            ),
            incident_gap_seconds=float(
                ALERT_POLICY["incidents"]["quiet_period_seconds"]
            ),
        )
        write_json(output / "sensitivity_manifest.json", {
            **LINEAGE,
            "dataset": DATASET,
            "labels_or_holdout_read": False,
            "summary": summary,
        })

sensitivity


## 3. Interpret stability, not development recall

A stable policy should preserve score ordering and produce a similar independent workload across reasonable sampling densities and seeds. This notebook does not select a model using fault labels. Large changes indicate that the production sample policy needs further investigation before Notebook 06 is treated as frozen.


In [ ]:
baseline = sensitivity.loc[
    sensitivity["rows_per_entity_day"].eq(
        str(MODEL_CONFIG["calibration_sample"]["maximum_rows_per_entity_day"])
    )
    & sensitivity["random_seed"].eq(policy["baseline_seed"])
].iloc[0]

baseline_workload = float(baseline["incidents_per_entity_day"])
comparison = sensitivity.assign(
    workload_ratio_to_baseline=(
        sensitivity["incidents_per_entity_day"] / baseline_workload
        if baseline_workload > 0 else np.nan
    ),
    threshold_ratio_to_baseline=sensitivity["threshold"] / baseline["threshold"],
)
display(comparison[[
    "rows_per_entity_day", "random_seed", "training_rows",
    "global_cap_applied", "score_rank_spearman_vs_baseline",
    "threshold_ratio_to_baseline", "incidents",
    "incidents_per_entity_day", "workload_ratio_to_baseline",
    "score_availability", "elapsed_seconds",
]])

rank_stability = pd.read_parquet(
    OUTPUT_ROOT / "sampling_rank_stability.parquet"
)
display(rank_stability.loc[
    rank_stability["left_configuration"].ne(
        rank_stability["right_configuration"]
    )
].nsmallest(12, "score_rank_spearman"))

print("No labels or holdout data were read.")
print("Next: run 06_PRIMARY_UNSUPERVISED_MODEL.ipynb with the registered policy.")
